![Module 6: Deploy to AgentCore Runtime](../images/module-6-deploy.svg)

# Module 6: Deploy to AgentCore Runtime

> Same `agent.py` → managed container on AWS with its own ARN. Invokable from Lambda, mobile, another agent — anywhere with AWS credentials.

📖 Full explainer: see the companion page in `content/06-deploy-agentcore/`.  
💻 Standalone deploy script: `code/step-05-deploy/deploy.sh`


## Retrieve configuration values

Run this cell to get the values you'll need for the terminal commands below.

In [ ]:
import os
import boto3

ssm = boto3.client('ssm')
MODEL_ID = ssm.get_parameter(Name='/aim308/model-id')['Parameter']['Value']
REGION = ssm.get_parameter(Name='/aim308/region')['Parameter']['Value']
MEMORY_ID = ssm.get_parameter(Name='/aim308/memory-id')['Parameter']['Value']
RUNTIME_ROLE_ARN = ssm.get_parameter(Name='/aim308/runtime-role-arn')['Parameter']['Value']
CODE_DIR = os.path.abspath('../code/step-05-deploy')

print('Copy these into your terminal:\n')
print(f'export AWS_REGION={REGION}')
print(f'export BEDROCK_AGENTCORE_MEMORY_ID={MEMORY_ID}')
print(f'export RUNTIME_ROLE_ARN={RUNTIME_ROLE_ARN}')
print(f'export CODE_DIR={CODE_DIR}')
print(f'\n# Model: {MODEL_ID}')

## Deploy Instructions

Open a **terminal** (File → New → Terminal in SageMaker Studio) and run the following commands.

### Step 0: Install the AgentCore CLI

The `@aws/agentcore` CLI requires Node 20+ and TypeScript.

```bash
# Install Node 20 via conda (works in SageMaker Studio)
conda install -y -c conda-forge nodejs=20

# Install the AgentCore CLI and TypeScript globally
npm install -g @aws/agentcore typescript@5

# Verify
agentcore --version
tsc --version
```

### Step 1: Set environment variables

Copy the `export` lines printed by the cell above, then paste them into your terminal.

### Step 2: Create the AgentCore project

```bash
cd aim308-workshop/notebooks/
agentcore create --name Aim308Nb \
    --no-agent \
    --defaults \
    --skip-install \
    --skip-python-setup \
    --skip-git
cd Aim308Nb
```

### Step 3: Register the agent

```bash
agentcore add agent \
    --name aim308_nb_agent \
    --type byo \
    --language Python \
    --framework Strands \
    --model-provider Bedrock \
    --code-location $CODE_DIR \
    --entrypoint agent.py \
    --protocol HTTP
```

### Step 4: Inject runtime env vars

```bash
cat >> agentcore/.env.local <<EOF
AWS_REGION=$AWS_REGION
BEDROCK_AGENTCORE_MEMORY_ID=$BEDROCK_AGENTCORE_MEMORY_ID
BEDROCK_AGENTCORE_MODEL_ID=$BEDROCK_AGENTCORE_MODEL_ID
EOF
```

### Step 4b: Set the execution role in agentcore.json

Run the cell below to patch the execution role into the generated config.


In [ ]:
import json

config_path = 'Aim308Nb/agentcore/agentcore.json'

with open(config_path) as f:
    config = json.load(f)

for rt in config.get('runtimes', []):
    if rt.get('name') == 'aim308_nb_agent':
        rt['executionRoleArn'] = RUNTIME_ROLE_ARN
        break

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print('✅ Updated agentcore.json with executionRoleArn')
print(json.dumps(config, indent=2))

### Step 5: Install CDK deps and deploy

```bash
# Install CDK dependencies (required before first deploy)
cd agentcore/cdk
npm install
cd ../..

# Deploy
agentcore deploy -y
```

Deployment takes 60–120 seconds.


## Invoke the deployed agent

Once deployed, test it from the same terminal:

```bash
# Basic invocation
agentcore invoke 'hello, introduce yourself and list your capabilities'

# Session continuity — pass --session-id to accumulate memory across turns
# Generate a session ID (must be 33+ chars)
export SESSION_ID=$(python3 -c "import uuid; print(uuid.uuid4().hex + uuid.uuid4().hex[:8])")
echo "Session ID: $SESSION_ID"

agentcore invoke --session-id $SESSION_ID 'I prefer python'
agentcore invoke --session-id $SESSION_ID 'what do you remember about me?'
```


## Observability

```bash
agentcore status
agentcore logs --since 5m
```


---

🎉 **Congrats — you built a fully autonomous, self-improving agent and deployed it to AWS.**

## Cleanup

Remove the deployed agent and its resources when you're done:

```bash
agentcore remove agent --name aim308_nb_agent
agentcore deploy -y   # CDK tears down the runtime
```
